In [1]:
import pandas as pd
from pathlib import Path

df = pd.read_csv(r"C:\Users\Vivek\Documents\dissertation\evaluation\quantitative_metrics.csv")
print(f"Total rows: {len(df)}")
print(f"Per spike: {df.groupby('spike').size().to_dict()}")

# Aggregate by the configuration variables that actually matter
group_cols = ["spike", "lora_scale", "ip_scale", "cn_scale"]
agg = df.groupby(group_cols, dropna=False).agg(
    n=("filename", "count"),
    clip_img=("clip_image_mean", "mean"),
    clip_txt=("clip_text_sim", "mean"),
    dinov2=("dinov2_style_mean", "mean"),
    lpips=("lpips_mean", "mean"),
).reset_index()

# Format nicely
def fmt(x):
    return f"{x:.4f}" if not pd.isna(x) else "  N/A "

print(f"\n{'Spike':<6} {'LoRA':>5} {'IP':>5} {'CN':>5} {'N':>4}  "
      f"{'CLIP-img':>9} {'CLIP-txt':>9} {'DINOv2':>9} {'LPIPS':>9}")
print("-" * 80)
for _, r in agg.iterrows():
    print(f"{r['spike']:<6} "
          f"{(r['lora_scale'] if not pd.isna(r['lora_scale']) else ''):>5} "
          f"{(r['ip_scale'] if not pd.isna(r['ip_scale']) else ''):>5} "
          f"{(r['cn_scale'] if not pd.isna(r['cn_scale']) else ''):>5} "
          f"{r['n']:>4}  "
          f"{fmt(r['clip_img']):>9} {fmt(r['clip_txt']):>9} "
          f"{fmt(r['dinov2']):>9} {fmt(r['lpips']):>9}")

# Per-tradition breakdown for the chosen v4 operating point
print("\n\nPer-tradition breakdown for v4 @ CN=0.4 (chosen operating point):")
v4_op = df[(df['spike'] == 'v4') & (df['cn_scale'] == 0.4)]
if len(v4_op) > 0:
    per_trad = v4_op.groupby('tradition').agg(
        n=("filename", "count"),
        clip_img=("clip_image_mean", "mean"),
        dinov2=("dinov2_style_mean", "mean"),
        lpips=("lpips_mean", "mean"),
    )
    print(per_trad.to_string())

Total rows: 123
Per spike: {'v2': 78, 'v3': 12, 'v4': 15, 'v5': 18}

Spike   LoRA    IP    CN    N   CLIP-img  CLIP-txt    DINOv2     LPIPS
--------------------------------------------------------------------------------
v2             0.5         36     0.4607    0.2522    0.3412    0.7772
v2             0.7         36     0.4209    0.2043    0.4285    0.7769
v2                          6     0.5462    0.2950      N/A     0.7981
v3       1.0   0.7          6     0.5130    0.2887    0.3158    0.7707
v3             0.7          6     0.4493    0.2408    0.3817    0.7421
v4       1.0   0.7   0.4    6     0.5521    0.3203    0.2451    0.7817
v4       1.0   0.7   0.5    6     0.6112    0.3183    0.1312    0.7739
v4                          3     0.5990    0.3126    0.1657    0.8280
v5       0.5   0.7   0.4    6     0.5450    0.3193    0.2700    0.7695
v5       0.7   0.7   0.4    6     0.5631    0.3219    0.2475    0.7708
v5       1.0   0.7   0.4    6     0.5521    0.3203    0.2451    0.781